In [134]:
import json
import requests
from pathlib import Path

import duckdb
import pandas as pd

In [135]:
DATA_BASE_PATH = Path("./data").resolve().absolute()
OSRM_BASE_URL = "http://localhost:5000"

In [136]:
gps_data_path = DATA_BASE_PATH / "gps_data.parquet"
ground_truth_path = DATA_BASE_PATH / "ground_truth_route.parquet"
newson_krumm_route_network_path = DATA_BASE_PATH / "road_network.parquet"

In [137]:
ground_truth_df = duckdb.query(
    f"""
    WITH gt AS (
        SELECT edge_id, traversed, ROW_NUMBER() OVER () as row_num
        FROM '{ground_truth_path}'
    )
    SELECT gt.edge_id, traversed, linestring
    FROM gt
    LEFT JOIN '{newson_krumm_route_network_path}' nkr 
    ON gt.edge_id = nkr.edge_id
    ORDER BY gt.row_num
    """
).to_df()
ground_truth_df

,edge_id,traversed,linestring
0,884147800801,1,"LINESTRING(-122.109748721123 47.6673012971878,..."
1,884147800802,1,"LINESTRING(-122.105398178101 47.6675292849541,..."
2,884147800421,1,"LINESTRING(-122.102048099041 47.6676607131958,..."
3,884147800422,1,"LINESTRING(-122.1028393507 47.6681300997734, -..."
4,884147800423,1,"LINESTRING(-122.103689610958 47.6685512065887,..."
...,...,...,...
571,884147801154,1,"LINESTRING(-122.14302957058 47.6376816630363, ..."
572,884147800845,1,"LINESTRING(-122.14302957058 47.6378801465034, ..."
573,884147800842,1,"LINESTRING(-122.14291960001 47.6386311650276, ..."
574,884147800843,1,"LINESTRING(-122.142908871174 47.6404094696045,..."


In [138]:
route_network_df = duckdb.query(
    f"""
    SELECT edge_id, linestring 
    FROM '{newson_krumm_route_network_path}'
    """
).to_df()
route_network_df

,edge_id,linestring
0,883991900000,"LINESTRING(-122.732318937778 47.8899192810059,..."
1,883991900001,"LINESTRING(-122.71107852459 47.8776508569717, ..."
2,883991900002,"LINESTRING(-122.707419991493 47.8761515021324,..."
3,883991900003,"LINESTRING(-122.707419991493 47.8761515021324,..."
4,883991900004,"LINESTRING(-122.715329825878 47.8818699717522,..."
...,...,...
158162,884152400184,"LINESTRING(-121.777908504009 47.4525502324104,..."
158163,884152400185,"LINESTRING(-121.781320273876 47.4532207846642,..."
158164,884152400186,"LINESTRING(-121.784308254719 47.4577805399895,..."
158165,884152400187,"LINESTRING(-121.782489717007 47.4503803253174,..."


In [139]:
def parse_linestring(linestring_series: pd.Series) -> pd.Series:
    track_segs = linestring_series.str.replace(r"^LINESTRING\(|\)$", "", regex=True)
    track_segs = track_segs.str.replace(",", ";").replace(r"\s+", " ", regex=True)
    track_segs = track_segs.str.split(";")
    track_segs = track_segs.apply(
        lambda x: [
            tuple((float(lat), float(lon)))
            for (lon, lat) in (point.split() for point in x)
        ]
    )
    return track_segs

In [140]:
ground_truth_df["track_segs"] = parse_linestring(ground_truth_df["linestring"])
route_network_df["track_segs"] = parse_linestring(route_network_df["linestring"])

In [141]:
gps_df = duckdb.query(f"SELECT * FROM '{gps_data_path}'").to_df()
gps_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7531 entries, 0 to 7530
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   recorded_timestamp  7531 non-null   datetime64[us]
 1   lon                 7531 non-null   float64       
 2   lat                 7531 non-null   float64       
dtypes: datetime64[us](1), float64(2)
memory usage: 176.6 KB


In [142]:
gps_df.head()

,recorded_timestamp,lon,lat
0,2009-01-17 20:27:37,-122.107083,47.667483
1,2009-01-17 20:27:38,-122.107067,47.667500
2,2009-01-17 20:27:39,-122.107067,47.667500
3,2009-01-17 20:27:40,-122.107033,47.667517
4,2009-01-17 20:27:41,-122.106983,47.667533


In [143]:
def request_osrm(gps_df: pd.DataFrame, output_path: Path) -> pd.DataFrame:
    points = gps_df[["lon", "lat", "recorded_timestamp"]]
    lat_lon_strs = points.apply(lambda row: f"{row['lon']},{row['lat']}", axis=1)[:20]
    coordinates = ";".join(lat_lon_strs)
    timestamps = ";".join(points["recorded_timestamp"][:20].astype(str))

    url = f"{OSRM_BASE_URL}/match/v1/car/{coordinates}?timestamps={timestamps}&annotations=true"

    req = requests.get(url)

    req.raise_for_status()

    with open(output_path, "wb") as f:
        f.write(req.content)

    print(f"Map matching result saved to {output_path}")

In [144]:
request_osrm(gps_df, DATA_BASE_PATH / "osrm-matched.json")

HTTPError: 400 Client Error: Bad Request for url: http://localhost:5000/match/v1/car/-122.1070833,47.66748333;-122.1070667,47.6675;-122.1070667,47.6675;-122.1070333,47.66751667;-122.1069833,47.66753333;-122.1069167,47.66753333;-122.1068333,47.66751667;-122.1067333,47.66751667;-122.1066,47.66751667;-122.1064667,47.66751667;-122.1063333,47.66751667;-122.1062,47.66751667;-122.1060667,47.66751667;-122.10595,47.66751667;-122.1058333,47.66751667;-122.1057,47.66751667;-122.1055833,47.66751667;-122.1054667,47.66751667;-122.10535,47.66751667;-122.1052167,47.66751667?timestamps=2009-01-17%2020:27:37;2009-01-17%2020:27:38;2009-01-17%2020:27:39;2009-01-17%2020:27:40;2009-01-17%2020:27:41;2009-01-17%2020:27:42;2009-01-17%2020:27:43;2009-01-17%2020:27:44;2009-01-17%2020:27:45;2009-01-17%2020:27:46;2009-01-17%2020:27:47;2009-01-17%2020:27:48;2009-01-17%2020:27:49;2009-01-17%2020:27:50;2009-01-17%2020:27:51;2009-01-17%2020:27:52;2009-01-17%2020:27:53;2009-01-17%2020:27:54;2009-01-17%2020:27:55;2009-01